# Lost-and-Found Matching System Testing Notebook

Use this notebook to test the matching prototype from VS Code or Jupyter. It runs the same logic as `run_evaluation.py`, but in a step-by-step notebook format.

Important: a notebook still needs a Python kernel. If VS Code asks you to select a kernel, choose Python 3.11+.

## 1. Load the Dataset

In [ ]:
from matching_system.data_loader import load_reports, load_labelled_pairs
from scripts.prepare_campus_data import prepare

prepare()  # Preserve the source CSV and repair six shifted rows in a copy.
reports = load_reports("data/processed/synthetic_campus_reports.csv")
labels = load_labelled_pairs("data/raw/campus/labelled_campus_pairs.csv")

lost_reports = [report for report in reports if report.report_type == "lost"]
found_reports = [report for report in reports if report.report_type == "found"]

print(f"Total reports: {len(reports)}")
print(f"Lost reports: {len(lost_reports)}")
print(f"Found reports: {len(found_reports)}")
print(f"Labelled pairs: {len(labels)}")

Expected output:

```text
Total reports: 46
Lost reports: 14
Found reports: 32
Labelled pairs: 32
```

## 2. Test One Known Lost Report

In [ ]:
from matching_system.matcher import rank_candidates

lost = next(report for report in lost_reports if report.report_id == "lost-001")
candidates = rank_candidates(lost, found_reports, limit=5, min_score=0)

print(lost.description)
print()
for candidate in candidates:
    print(candidate.found_report_id, candidate.score)
    print("  shared:", candidate.shared_characteristics)
    print("  conflicts:", candidate.conflicting_details)
    print("  explanation:", candidate.explanation)
    print()

For `lost-001`, the best candidate should be `found-001` because it shares the backpack category, dark/black wording, red zipper, notebook detail, nearby library location, and close time.

## 3. Run Evaluation Metrics

In [ ]:
from matching_system.evaluation import evaluate

metrics = evaluate(reports, labels, k=3)

print("Precision@3:", metrics["precision_at_k"])
print("Recall@3:", metrics["recall_at_k"])
print("Mean reciprocal rank:", metrics["mean_reciprocal_rank"])
print("Labelled distractors in top 3:", len(metrics["false_positives"]))
print("Unlabelled candidates in top 3:", len(metrics["unlabelled_candidates"]))

## 4. Inspect False Positives

In [ ]:
for item in metrics["false_positives"][:15]:
    print(item)

False positives are useful at this stage. They show where the matcher is over-trusting generic similarity, weak location overlap, or incomplete descriptions.

## 5. Manual Smoke Tests

In [ ]:
def assert_top_match(lost_id, expected_found_id):
    lost_report = next(report for report in lost_reports if report.report_id == lost_id)
    ranked = rank_candidates(lost_report, found_reports, limit=3, min_score=0)
    actual = ranked[0].found_report_id
    assert actual == expected_found_id, f"Expected {expected_found_id}, got {actual}"
    print(f"PASS: {lost_id} top match is {expected_found_id}")

assert_top_match("lost-001", "found-001")
assert_top_match("lost-002", "found-004")
assert_top_match("lost-003", "found-007")

## 6. Image Matching Status

In [ ]:
from matching_system.image_matching import image_matching_status

image_matching_status()